# Notebook 02.5: Manual Identifier Mapping Helper

## Purpose

This helper notebook supports manual completion and validation of the company identifier mapping file created by Notebook 02. It does not scrape websites, fetch historical prices, calculate returns, or build the dashboard. Its only purpose is to prepare a verified manual mapping file before Notebook 03.

After editing and saving the manual mapping file, rerun Notebook 02 so that `clean_events_with_identifiers_v01.csv` is updated with the verified mapping fields.

## Manual Verification Instructions

Use the generated search/reference URLs to verify identifiers manually. Prefer identifiers from reliable exchange, Yahoo Finance, Google Finance / Google Sheets, TASE, Maya, or other clearly attributable market-reference pages.

Do not guess tickers from Hebrew company names. Israeli Yahoo Finance tickers usually end with `.TA`, and Google Finance / Google Sheets symbols usually start with `TLV:`, but both require verification. If you are unsure, leave the identifier field empty and add a note in `mapping_notes`.

Editable fields in this helper are:

- `yahoo_ticker`
- `google_finance_symbol`
- `tase_security_id`
- `isin`
- `mapping_notes`

The URL columns are preserved for reference and should not be treated as verified identifiers.

## 1. Imports and Configuration

In [1]:
import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == "notebooks" else Path.cwd().resolve()
DATA_INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
DATA_OUTPUT_DIR = PROJECT_ROOT / "data" / "output"

DATA_INTERIM_DIR.mkdir(parents=True, exist_ok=True)
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

manual_review_path = DATA_INTERIM_DIR / "company_identifier_mapping_manual_review_v01.xlsx"
current_mapping_path = DATA_INTERIM_DIR / "company_identifier_mapping_v01.xlsx"
manual_output_path = DATA_INTERIM_DIR / "company_identifier_mapping_manual.xlsx"
validation_csv_path = DATA_OUTPUT_DIR / "manual_mapping_validation_report_v01.csv"
validation_json_path = DATA_OUTPUT_DIR / "manual_mapping_validation_report_v01.json"

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Users\Guy\Desktop\Tene


## 2. Load Manual Review Inputs

The required input is the manual-review workbook from Notebook 02. If the current full mapping table is available, it is also loaded for context and to preserve any previously filled identifiers.

In [2]:
if not manual_review_path.exists():
    raise FileNotFoundError(
        f"Manual review workbook not found: {manual_review_path.relative_to(PROJECT_ROOT)}. "
        "Run Notebook 02 before using this helper."
    )

manual_review_df = pd.read_excel(manual_review_path, dtype=object)

if current_mapping_path.exists():
    current_mapping_df = pd.read_excel(current_mapping_path, dtype=object)
else:
    current_mapping_df = pd.DataFrame()
    warnings.warn("company_identifier_mapping_v01.xlsx was not found; continuing with manual review workbook only.")

print(f"Manual review rows: {len(manual_review_df)}")
print(f"Current mapping rows: {len(current_mapping_df)}")
manual_review_df.head()

Manual review rows: 49
Current mapping rows: 49


,company_name_original,company_name_normalized,yahoo_ticker,google_finance_symbol,tase_security_id,isin,google_search_url,maya_search_url,tase_search_url,bizportal_search_url,investing_search_url,globes_search_url,mapping_notes
0,א. לוי השקעות,א. לוי השקעות,NaN,NaN,NaN,NaN,https://www.google.com/search?q=%D7%90.+%D7%9C...,https://www.google.com/search?q=%D7%90.+%D7%9C...,https://www.google.com/search?q=%D7%90.+%D7%9C...,https://www.google.com/search?q=%D7%90.+%D7%9C...,https://www.google.com/search?q=%D7%90.+%D7%9C...,https://www.google.com/search?q=%D7%90.+%D7%9C...,Identifier verification required before price ...
1,אבו מגורים,אבו מגורים,NaN,NaN,NaN,NaN,https://www.google.com/search?q=%D7%90%D7%91%D...,https://www.google.com/search?q=%D7%90%D7%91%D...,https://www.google.com/search?q=%D7%90%D7%91%D...,https://www.google.com/search?q=%D7%90%D7%91%D...,https://www.google.com/search?q=%D7%90%D7%91%D...,https://www.google.com/search?q=%D7%90%D7%91%D...,Identifier verification required before price ...
2,אגוד הנפקות,אגוד הנפקות,NaN,NaN,NaN,NaN,https://www.google.com/search?q=%D7%90%D7%92%D...,https://www.google.com/search?q=%D7%90%D7%92%D...,https://www.google.com/search?q=%D7%90%D7%92%D...,https://www.google.com/search?q=%D7%90%D7%92%D...,https://www.google.com/search?q=%D7%90%D7%92%D...,https://www.google.com/search?q=%D7%90%D7%92%D...,Identifier verification required before price ...
3,אוברסיז,אוברסיז,NaN,NaN,NaN,NaN,https://www.google.com/search?q=%D7%90%D7%95%D...,https://www.google.com/search?q=%D7%90%D7%95%D...,https://www.google.com/search?q=%D7%90%D7%95%D...,https://www.google.com/search?q=%D7%90%D7%95%D...,https://www.google.com/search?q=%D7%90%D7%95%D...,https://www.google.com/search?q=%D7%90%D7%95%D...,Identifier verification required before price ...
4,אוטונומוס,אוטונומוס,NaN,NaN,NaN,NaN,https://www.google.com/search?q=%D7%90%D7%95%D...,https://www.google.com/search?q=%D7%90%D7%95%D...,https://www.google.com/search?q=%D7%90%D7%95%D...,https://www.google.com/search?q=%D7%90%D7%95%D...,https://www.google.com/search?q=%D7%90%D7%95%D...,https://www.google.com/search?q=%D7%90%D7%95%D...,Identifier verification required before price ...


## 3. Prepare Editable Manual Mapping Table

The table below keeps one row per company requiring manual review. Identifier fields are editable. Search and source-reference URL columns are preserved to support manual verification.

In [3]:
EDITABLE_COLUMNS = [
    "yahoo_ticker",
    "google_finance_symbol",
    "tase_security_id",
    "isin",
    "mapping_notes",
]

REFERENCE_URL_COLUMNS = [
    "google_search_url",
    "maya_search_url",
    "tase_search_url",
    "bizportal_search_url",
    "investing_search_url",
    "globes_search_url",
]

BASE_COLUMNS = [
    "company_name_original",
    "company_name_normalized",
]

REQUIRED_MANUAL_COLUMNS = BASE_COLUMNS + EDITABLE_COLUMNS + REFERENCE_URL_COLUMNS

missing_review_columns = [column for column in REQUIRED_MANUAL_COLUMNS if column not in manual_review_df.columns]
if missing_review_columns:
    raise ValueError(f"Manual review workbook is missing required columns: {missing_review_columns}")

manual_work_df = manual_review_df[REQUIRED_MANUAL_COLUMNS].copy()

if not current_mapping_df.empty and "company_name_original" in current_mapping_df.columns:
    current_subset_cols = ["company_name_original"] + [col for col in EDITABLE_COLUMNS if col in current_mapping_df.columns]
    current_subset = current_mapping_df[current_subset_cols].drop_duplicates("company_name_original", keep="last")
    manual_work_df = manual_work_df.merge(
        current_subset,
        on="company_name_original",
        how="left",
        suffixes=("", "__current"),
    )
    for column in EDITABLE_COLUMNS:
        current_column = f"{column}__current"
        if current_column in manual_work_df.columns:
            current_values = manual_work_df[current_column]
            original_values = manual_work_df[column]
            manual_work_df[column] = np.where(
                current_values.notna() & current_values.astype(str).str.strip().ne(""),
                current_values,
                original_values,
            )
            manual_work_df = manual_work_df.drop(columns=[current_column])

for column in EDITABLE_COLUMNS:
    manual_work_df[column] = manual_work_df[column].fillna("").astype(str)
    manual_work_df.loc[manual_work_df[column].str.lower().isin(["nan", "none", "null"]), column] = ""

manual_work_df = manual_work_df.sort_values("company_name_original").reset_index(drop=True)
display(manual_work_df)

,company_name_original,company_name_normalized,yahoo_ticker,google_finance_symbol,tase_security_id,isin,mapping_notes,google_search_url,maya_search_url,tase_search_url,bizportal_search_url,investing_search_url,globes_search_url
0,א. לוי השקעות,א. לוי השקעות,,,,,Identifier verification required before price ...,https://www.google.com/search?q=%D7%90.+%D7%9C...,https://www.google.com/search?q=%D7%90.+%D7%9C...,https://www.google.com/search?q=%D7%90.+%D7%9C...,https://www.google.com/search?q=%D7%90.+%D7%9C...,https://www.google.com/search?q=%D7%90.+%D7%9C...,https://www.google.com/search?q=%D7%90.+%D7%9C...
1,אבו מגורים,אבו מגורים,,,,,Identifier verification required before price ...,https://www.google.com/search?q=%D7%90%D7%91%D...,https://www.google.com/search?q=%D7%90%D7%91%D...,https://www.google.com/search?q=%D7%90%D7%91%D...,https://www.google.com/search?q=%D7%90%D7%91%D...,https://www.google.com/search?q=%D7%90%D7%91%D...,https://www.google.com/search?q=%D7%90%D7%91%D...
2,אגוד הנפקות,אגוד הנפקות,,,,,Identifier verification required before price ...,https://www.google.com/search?q=%D7%90%D7%92%D...,https://www.google.com/search?q=%D7%90%D7%92%D...,https://www.google.com/search?q=%D7%90%D7%92%D...,https://www.google.com/search?q=%D7%90%D7%92%D...,https://www.google.com/search?q=%D7%90%D7%92%D...,https://www.google.com/search?q=%D7%90%D7%92%D...
3,אוברסיז,אוברסיז,,,,,Identifier verification required before price ...,https://www.google.com/search?q=%D7%90%D7%95%D...,https://www.google.com/search?q=%D7%90%D7%95%D...,https://www.google.com/search?q=%D7%90%D7%95%D...,https://www.google.com/search?q=%D7%90%D7%95%D...,https://www.google.com/search?q=%D7%90%D7%95%D...,https://www.google.com/search?q=%D7%90%D7%95%D...
4,אוטונומוס,אוטונומוס,,,,,Identifier verification required before price ...,https://www.google.com/search?q=%D7%90%D7%95%D...,https://www.google.com/search?q=%D7%90%D7%95%D...,https://www.google.com/search?q=%D7%90%D7%95%D...,https://www.google.com/search?q=%D7%90%D7%95%D...,https://www.google.com/search?q=%D7%90%D7%95%D...,https://www.google.com/search?q=%D7%90%D7%95%D...
5,איי ארגנטו,איי ארגנטו,,,,,Identifier verification required before price ...,https://www.google.com/search?q=%D7%90%D7%99%D...,https://www.google.com/search?q=%D7%90%D7%99%D...,https://www.google.com/search?q=%D7%90%D7%99%D...,https://www.google.com/search?q=%D7%90%D7%99%D...,https://www.google.com/search?q=%D7%90%D7%99%D...,https://www.google.com/search?q=%D7%90%D7%99%D...
6,אימקו,אימקו,,,,,Identifier verification required before price ...,https://www.google.com/search?q=%D7%90%D7%99%D...,https://www.google.com/search?q=%D7%90%D7%99%D...,https://www.google.com/search?q=%D7%90%D7%99%D...,https://www.google.com/search?q=%D7%90%D7%99%D...,https://www.google.com/search?q=%D7%90%D7%99%D...,https://www.google.com/search?q=%D7%90%D7%99%D...
7,אינרום בניה,אינרום בניה,,,,,Identifier verification required before price ...,https://www.google.com/search?q=%D7%90%D7%99%D...,https://www.google.com/search?q=%D7%90%D7%99%D...,https://www.google.com/search?q=%D7%90%D7%99%D...,https://www.google.com/search?q=%D7%90%D7%99%D...,https://www.google.com/search?q=%D7%90%D7%99%D...,https://www.google.com/search?q=%D7%90%D7%99%D...
8,"אלקטרה נדל""ן","אלקטרה נדל""ן",,,,,Identifier verification required before price ...,https://www.google.com/search?q=%D7%90%D7%9C%D...,https://www.google.com/search?q=%D7%90%D7%9C%D...,https://www.google.com/search?q=%D7%90%D7%9C%D...,https://www.google.com/search?q=%D7%90%D7%9C%D...,https://www.google.com/search?q=%D7%90%D7%9C%D...,https://www.google.com/search?q=%D7%90%D7%9C%D...
9,"אמיליה פיתוח )מ.עו.פ.( בע""מ","אמיליה פיתוח )מ.עו.פ.( בע""מ",,,,,Identifier verification required before price ...,https://www.google.com/search?q=%D7%90%D7%9E%D...,https://www.google.com/search?q=%D7%90%D7%9E%D...,https://www.google.com/search?q=%D7%90%D7%9E%D...,https://www.google.com/search?q=%D7%90%D7%9E%D...,https://www.google.com/search?q=%D7%90%D7%9E%D...,

## 4. Lightweight Validation

Validation is intentionally conservative. A row is considered ready only if at least one of `yahoo_ticker`, `google_finance_symbol`, `tase_security_id`, or `isin` is filled. Format warnings do not block saving, but they should be reviewed before rerunning Notebook 02.

In [4]:
def clean_text(value):
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if text.lower() in {"nan", "none", "null"}:
        return ""
    return text


def has_documented_reason(notes):
    note = clean_text(notes).lower()
    return bool(note) and any(token in note for token in ["exception", "חריג", "reason", "verified", "manual"])


def validate_manual_mapping_row(row):
    yahoo_ticker = clean_text(row.get("yahoo_ticker", ""))
    google_symbol = clean_text(row.get("google_finance_symbol", ""))
    tase_security_id = clean_text(row.get("tase_security_id", ""))
    isin = clean_text(row.get("isin", ""))
    notes = clean_text(row.get("mapping_notes", ""))

    issues = []
    warnings_list = []

    if yahoo_ticker and not yahoo_ticker.upper().endswith(".TA") and not has_documented_reason(notes):
        warnings_list.append("YAHOO_TICKER_UNUSUAL_SUFFIX")

    if google_symbol and not google_symbol.upper().startswith("TLV:") and not has_documented_reason(notes):
        warnings_list.append("GOOGLE_FINANCE_SYMBOL_UNUSUAL_PREFIX")

    if isin and not re.fullmatch(r"[A-Z]{2}[A-Z0-9]{9}[0-9]", isin.upper()):
        warnings_list.append("ISIN_FORMAT_REVIEW_RECOMMENDED")

    has_identifier = any([yahoo_ticker, google_symbol, tase_security_id, isin])
    if not has_identifier:
        issues.append("MISSING_ALL_IDENTIFIERS")

    return pd.Series({
        "has_any_identifier": has_identifier,
        "ready_for_price_extraction": has_identifier and not issues,
        "validation_issues": "; ".join(issues),
        "validation_warnings": "; ".join(warnings_list),
    })


validation_columns_df = manual_work_df.apply(validate_manual_mapping_row, axis=1)
manual_validated_df = pd.concat([manual_work_df, validation_columns_df], axis=1)

manual_validated_df[[
    "company_name_original",
    "yahoo_ticker",
    "google_finance_symbol",
    "tase_security_id",
    "isin",
    "has_any_identifier",
    "ready_for_price_extraction",
    "validation_issues",
    "validation_warnings",
]].head(20)

,company_name_original,yahoo_ticker,google_finance_symbol,tase_security_id,isin,has_any_identifier,ready_for_price_extraction,validation_issues,validation_warnings
0,א. לוי השקעות,,,,,False,False,MISSING_ALL_IDENTIFIERS,
1,אבו מגורים,,,,,False,False,MISSING_ALL_IDENTIFIERS,
2,אגוד הנפקות,,,,,False,False,MISSING_ALL_IDENTIFIERS,
3,אוברסיז,,,,,False,False,MISSING_ALL_IDENTIFIERS,
4,אוטונומוס,,,,,False,False,MISSING_ALL_IDENTIFIERS,
5,איי ארגנטו,,,,,False,False,MISSING_ALL_IDENTIFIERS,
6,אימקו,,,,,False,False,MISSING_ALL_IDENTIFIERS,
7,אינרום בניה,,,,,False,False,MISSING_ALL_IDENTIFIERS,
8,"אלקטרה נדל""ן",,,,,False,False,MISSING_ALL_IDENTIFIERS,
9,"אמיליה פיתוח )מ.עו.פ.( בע""מ",,,,,False,False,MISSING_ALL_IDENTIFIERS,


## 5. Mapping Readiness Summary

This summary shows how much manual mapping work remains before Notebook 03 can extract prices. It should improve as verified identifiers are filled into the manual workbook.

In [5]:
summary = {
    "total_companies": int(len(manual_validated_df)),
    "companies_with_yahoo_ticker": int(manual_validated_df["yahoo_ticker"].apply(clean_text).ne("").sum()),
    "companies_with_google_finance_symbol": int(manual_validated_df["google_finance_symbol"].apply(clean_text).ne("").sum()),
    "companies_with_tase_security_id": int(manual_validated_df["tase_security_id"].apply(clean_text).ne("").sum()),
    "companies_with_isin": int(manual_validated_df["isin"].apply(clean_text).ne("").sum()),
    "companies_still_missing_all_identifiers": int((~manual_validated_df["has_any_identifier"]).sum()),
    "percentage_ready_for_price_extraction": round(float(manual_validated_df["ready_for_price_extraction"].mean() * 100), 2) if len(manual_validated_df) else 0.0,
}

summary_df = pd.DataFrame([{"metric": key, "value": value} for key, value in summary.items()])
summary_df

,metric,value
0,total_companies,49.0
1,companies_with_yahoo_ticker,0.0
2,companies_with_google_finance_symbol,0.0
3,companies_with_tase_security_id,0.0
4,companies_with_isin,0.0
5,companies_still_missing_all_identifiers,49.0
6,percentage_ready_for_price_extraction,0.0


## 6. Save Manual Mapping Workbook and Validation Reports

The saved workbook is the file the user should edit:

`data/interim/company_identifier_mapping_manual.xlsx`

After filling identifiers and notes, rerun this helper to refresh validation, then rerun Notebook 02 to update the event-level identifier table.

In [6]:
manual_output_columns = REQUIRED_MANUAL_COLUMNS
manual_output_df = manual_work_df[manual_output_columns].copy()
manual_output_df.to_excel(manual_output_path, index=False)

try:
    from openpyxl import load_workbook

    workbook = load_workbook(manual_output_path)
    worksheet = workbook.active
    worksheet.freeze_panes = "A2"
    for column_cells in worksheet.columns:
        max_length = max(len(str(cell.value)) if cell.value is not None else 0 for cell in column_cells)
        worksheet.column_dimensions[column_cells[0].column_letter].width = min(max(max_length + 2, 14), 75)
    workbook.save(manual_output_path)
except Exception as exc:
    warnings.warn(f"Manual workbook was saved, but formatting could not be applied: {exc}")

validation_report_df = manual_validated_df[[
    "company_name_original",
    "yahoo_ticker",
    "google_finance_symbol",
    "tase_security_id",
    "isin",
    "has_any_identifier",
    "ready_for_price_extraction",
    "validation_issues",
    "validation_warnings",
    "mapping_notes",
]].copy()
validation_report_df.to_csv(validation_csv_path, index=False, encoding="utf-8-sig")
validation_json_path.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2, sort_keys=True),
    encoding="utf-8",
)

pd.DataFrame({
    "saved_file": [
        str(manual_output_path.relative_to(PROJECT_ROOT)),
        str(validation_csv_path.relative_to(PROJECT_ROOT)),
        str(validation_json_path.relative_to(PROJECT_ROOT)),
    ]
})

,saved_file
0,data\interim\company_identifier_mapping_manual...
1,data\output\manual_mapping_validation_report_v...
2,data\output\manual_mapping_validation_report_v...


## 7. Validation Checks

The helper should preserve all companies requiring manual review and save the manual workbook plus validation reports. These checks do not require identifiers to be filled yet.

In [7]:
if len(manual_output_df) != len(manual_review_df):
    raise ValueError(
        f"Manual output row count changed: review={len(manual_review_df)}, output={len(manual_output_df)}"
    )

missing_output_columns = [column for column in REQUIRED_MANUAL_COLUMNS if column not in manual_output_df.columns]
if missing_output_columns:
    raise ValueError(f"Manual output is missing required columns: {missing_output_columns}")

for path in [manual_output_path, validation_csv_path, validation_json_path]:
    if not path.exists():
        raise FileNotFoundError(f"Expected output was not created: {path}")

validation_check_summary = {
    "manual_review_rows_loaded": int(len(manual_review_df)),
    "manual_output_rows_saved": int(len(manual_output_df)),
    "manual_output_exists": manual_output_path.exists(),
    "validation_csv_exists": validation_csv_path.exists(),
    "validation_json_exists": validation_json_path.exists(),
}
validation_check_summary

{'manual_review_rows_loaded': 49,
 'manual_output_rows_saved': 49,
 'manual_output_exists': True,
 'validation_csv_exists': True,
 'validation_json_exists': True}

## Notebook 02.5 Conclusions

This helper created an editable manual mapping workbook and validation reports. It intentionally did not scrape websites, fetch historical prices, or calculate returns.

Next workflow:

1. Open `data/interim/company_identifier_mapping_manual.xlsx`.
2. Use the search/reference URLs to manually verify identifiers.
3. Fill `yahoo_ticker`, `google_finance_symbol`, `tase_security_id`, and/or `isin` where verified.
4. Add notes for uncertainty or unusual symbol formats.
5. Rerun this helper to refresh validation.
6. Rerun Notebook 02 so `clean_events_with_identifiers_v01.csv` is updated with verified identifiers.